# 1. Library calling

In [17]:
import pandas as pd
import warnings
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [18]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][60]
print(search_text)
Source="Walmart"
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
filename='_Link.txt'
f=open(IFolder+'\\'+search_text+filename,'r')
Linebraker='w-25'
prefix='https://www.walmart.com//ip/'

                           Product Name
0                         Ignition Coil
1               Windshield Washer Pumps
2                 coupler trailer locks
3   Adjustable Trailer Hitch Ball Mount
4                        Vacuum Cleaner
..                                  ...
56                  Windshield Sunshade
57                     Motor Cycle Bags
58                         Brake Fluids
59                   CAR Vacuum Cleaner
60                 Power Steering Fluid

[61 rows x 1 columns]
Power Steering Fluid


# 3. Setting Webdriver and Website Specific Information

In [19]:
content=f.read()

In [20]:
List=content.split(Linebraker)

In [21]:
print(len(List))

518


# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [36]:
cols =['Sl.No'] #,'Review_Mentions'
df = pd.DataFrame(columns=cols)

In [23]:
List[2].split('current price')[1].split ('</span>')[0].replace('Now',"").replace('$',"")

'  9.12'

In [24]:
List[2].split('end mt1">')[1].split ('</div>')[0]

'28.5 ¢/fl oz'

In [37]:
count=0

for i in range(1,len(List)):
    df.loc[count,'Sl.No']=i
    try:
        df.loc[count,'Current Price']=List[i].split('current price')[1].split ('</span>')[0].replace('Now',"").replace('$',"").replace(",","")
    except:
        pass
    try:
        df.loc[count,'List Price']=List[i].split('<div class="gray mr1 strike f6 f4-l flex items-end mt1" aria-hidden="true">')[1].split('</div>')[0].replace('$',"").replace(",","")
    except:
        try:
            df.loc[count,'List Price']=List[i].split('current price')[1].split ('</span>')[0].replace('Now',"").replace('$',"").replace(' ',"").replace(",","")
        except:
            pass
    Rating=List[i].split(' out of 5 Stars')[0].rsplit('<span class="w_iUH7">')[-1]
    if len(Rating)>3:
        df.loc[count,'Rating']=0
    else:
        df.loc[count,'Rating']=Rating
    try:
        df.loc[count,'No of Ratings']=List[i].split(' review')[0].split('5 Stars. ')[1]
    except:
        df.loc[count,'No of Ratings']=0
    try:
        df.loc[count,'Unit Price']=List[i].split('end mt1">')[1].split ('</div>')[0]
    except:
        df.loc[count,'Unit Price']=""
    df.loc[count,'Name']=List[i].split(' f6 f5-l lh-copy">')[1].split('</span>')[0]
    try:
        link=List[i].split('''%2Fip%2F''')[1].split('''%3F''')[0].replace('''%2F''','''/''')
    except:
        link=List[i].split('''target="" href="/ip/''')[1].split('''?''')[0]
    df.loc[count,'Links']=prefix+link
    df.loc[count,'Source']=Source
    df.loc[count,'Product']=search_text
    count=count+1

In [38]:
df

,Sl.No,Current Price,List Price,Rating,No of Ratings,Unit Price,Name,Links,Source,Product
0,1,7.68,7.68,4.7,395,24.0 ¢/fl oz,"Super Tech Power Steering Fluid, 32 oz",https://www.walmart.com//ip/Super-Tech-Power-S...,Walmart,Power Steering Fluid
1,2,9.12,10.49,4.6,251,28.5 ¢/fl oz,"Prestone Power Steering Fluid Plus Stop Leak, ...",https://www.walmart.com//ip/Prestone-Power-Ste...,Walmart,Power Steering Fluid
2,3,3.22,3.22,4.7,202,26.8 ¢/fl oz,"Prestone Universal Power Steering Fluid, 12 fl oz",https://www.walmart.com//ip/Prestone-Universal...,Walmart,Power Steering Fluid
3,4,7.88,7.88,4.7,72,24.6 ¢/fl oz,Prestone Universal Power Steering Fluid - 32 f...,https://www.walmart.com//ip/Prestone-Universal...,Walmart,Power Steering Fluid
4,5,5.73,10.75,4.8,74,47.8 ¢/fl oz,"Prestone Power Steering Fluid Plus Stop Leak, ...",https://www.walmart.com//ip/Prestone-Power-Ste...,Walmart,Power Steering Fluid
...,...,...,...,...,...,...,...,...,...,...
512,513,31.71,31.71,0,0,,Red Line (30404) Power Steering Fluid - 1 Quart,https://www.walmart.com//ip/Red-Line-30404-Pow...,Walmart,Power Steering Fluid
513,514,27.76,27.76,0,0,,"Armored 65464 STP Power Steering Fluid, 32 oz",https://www.walmart.com//ip/Armored-65464-STP-...,Walmart,Power Steering Fluid
514,515,17.04,17.04,0,0,,Gunk M2732 Power Steering Fluid - 1 Quart,https://www.walmart.com//ip/Gunk-M2732-Power-S...,Walmart,Power Steering Fluid
515,516,37.38,37.38,0,0,,"PSF 10032001 Power Steering Fluid (KRC Quart),...",https://www.walmart.com//ip/PSF-10032001-Power...,Walmart,Power Steering Fluid


# 5. Post Processing Data and Exporting

In [39]:
df['Sl.No']=df['Sl.No'].astype(int)
df['List Price']=(df['List Price']).astype(float)
df['Current Price']=(df['Current Price']).astype(float)
df['Rating']=(df['Rating']).astype(float)
df['No of Ratings']=(df['No of Ratings']).astype(float)

In [40]:
cols=["Sl.No",
"Name",
"Product",
"Current Price",
"List Price",
"Rating",
"No of Ratings",
"Links",
"Source"
]
df = df[cols]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 517 entries, 0 to 516
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sl.No          517 non-null    int32  
 1   Name           517 non-null    object 
 2   Product        517 non-null    object 
 3   Current Price  501 non-null    float64
 4   List Price     501 non-null    float64
 5   Rating         517 non-null    float64
 6   No of Ratings  517 non-null    float64
 7   Links          517 non-null    object 
 8   Source         517 non-null    object 
dtypes: float64(4), int32(1), object(4)
memory usage: 54.5+ KB


In [41]:
df.to_excel(OFolder+'\\'+f'{Source}ProductDetails_'+search_text+'.xlsx', index=False)

# 99. Archived Codes